In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2
import matplotlib.pyplot as plt
import warnings

# Menonaktifkan peringatan
warnings.filterwarnings("ignore")


In [ ]:
# 1. Membaca dataset
file_path = '/content/drive/MyDrive/Pemrograman data mining/kpop_rankings.csv'
kpop_data = pd.read_csv(file_path)

# 2. Mengganti nilai NaN pada kolom yang mungkin kosong
kpop_data.fillna("", inplace=True)

# Mengonversi kolom 'song_title' dan 'artist' menjadi fitur yang dapat digunakan oleh model
X = kpop_data['song_title'] + " " + kpop_data['artist']
y = kpop_data['rank']  # Peringkat lagu sebagai target

# 3. Mengonversi teks menjadi vektor menggunakan TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X_tfidf = vectorizer.fit_transform(X)

# 4. Encoding target rank untuk klasifikasi
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 5. Membagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y_encoded, test_size=0.3, random_state=42)

# 6. Membuat model RandomForest
model = RandomForestClassifier(n_estimators=100, random_state=42)

# 7. Latih model tanpa seleksi fitur
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_no_selection = accuracy_score(y_test, y_pred)
print(f"Akurasi tanpa seleksi fitur: {accuracy_no_selection:.4f}")

# 8. Seleksi Fitur dengan Mutual Information (MI) - Menggunakan 1000 fitur
selector_mi = SelectKBest(mutual_info_classif, k=1000)
X_train_mi = selector_mi.fit_transform(X_train, y_train)
X_test_mi = selector_mi.transform(X_test)

# 9. Latih model dengan data yang sudah diseleksi fiturnya (Mutual Information)
model.fit(X_train_mi, y_train)
y_pred_mi = model.predict(X_test_mi)
accuracy_mi = accuracy_score(y_test, y_pred_mi)
print(f"Akurasi dengan Mutual Information: {accuracy_mi:.4f}")

# 10. Seleksi Fitur dengan Chi-Square Test (χ²) - Menggunakan 1000 fitur
selector_chi2 = SelectKBest(chi2, k=1000)
X_train_chi2 = selector_chi2.fit_transform(X_train, y_train)
X_test_chi2 = selector_chi2.transform(X_test)

# 11. Latih model dengan data yang sudah diseleksi fiturnya (Chi-Square)
model.fit(X_train_chi2, y_train)
y_pred_chi2 = model.predict(X_test_chi2)
accuracy_chi2 = accuracy_score(y_test, y_pred_chi2)
print(f"Akurasi dengan Chi-Square Test: {accuracy_chi2:.4f}")

# 12. Perbandingan Akurasi
comparison_data = {
    'Without Feature Selection': [accuracy_no_selection],
    'With Mutual Information': [accuracy_mi],
    'With Chi-Square Test': [accuracy_chi2]
}

comparison_data_df = pd.DataFrame(comparison_data)

# Tampilkan perbandingan akurasi
comparison_data_df.plot(kind='bar', title="Perbandingan Akurasi dengan Seleksi Fitur yang Berbeda", legend=True)
plt.ylabel('Akurasi')
plt.xticks(rotation=0)
plt.show()

# 13. Fitur Terpilih berdasarkan Mutual Information dan Chi-Square Test
selected_features_mi = selector_mi.get_support(indices=True)
selected_feature_names_mi = [vectorizer.get_feature_names_out()[i] for i in selected_features_mi]

selected_features_chi2 = selector_chi2.get_support(indices=True)
selected_feature_names_chi2 = [vectorizer.get_feature_names_out()[i] for i in selected_features_chi2]

print("\nFitur terpilih berdasarkan Mutual Information:")
print(selected_feature_names_mi[:20])  # Menampilkan 20 fitur teratas yang dipilih oleh MI

print("\nFitur terpilih berdasarkan Chi-Square Test:")
print(selected_feature_names_chi2[:20])  # Menampilkan 20 fitur teratas yang dipilih oleh Chi-Square1
